# Supabase token validation (Notebook)

Este notebook demonstra duas abordagens para validar tokens de acesso (access_token) do Supabase:

- Abordagem recomendada: o notebook chama o seu backend (`/admin/verify-session` ou `/admin/alunos`) que possui a `service_role` (secret) e valida o token com segurança.
- Abordagem direta (apenas para testes locais): o notebook chama o endpoint `/auth/v1/user` do Supabase usando a `service_role` (NÃO FAÇA ISSO em produção).

Preencha um arquivo `.env` local com `SUPABASE_URL` e `SUPABASE_ANON_KEY` (e opcionalmente `SUPABASE_SECRET_KEY` **somente para testes locais**).

In [ ]:
# Instala dependências (executar apenas se necessário)
!pip install -q python-dotenv requests

In [ ]:
# Carrega variáveis de ambiente de .env (local only)
import os
from dotenv import load_dotenv

load_dotenv('.env')  # crie .env localmente a partir de .env.example

SUPABASE_URL = os.getenv('SUPABASE_URL')
ANON_KEY = os.getenv('SUPABASE_ANON_KEY')
SERVICE_KEY = os.getenv('SUPABASE_SECRET_KEY')  # opcional, apenas para testes locais
ADMIN_SERVER = os.getenv('ADMIN_SERVER', 'http://localhost:3000')

print('SUPABASE_URL:', bool(SUPABASE_URL))
print('ANON_KEY:', bool(ANON_KEY))
print('SERVICE_KEY (present?):', bool(SERVICE_KEY))
print('ADMIN_SERVER:', ADMIN_SERVER)

In [ ]:
# 1) Fazer sign-in usando a API pública para obter access_token (email/password)
import getpass
import requests

if not SUPABASE_URL or not ANON_KEY:
    raise RuntimeError('Defina SUPABASE_URL e SUPABASE_ANON_KEY no .env')

email = os.getenv('TEST_EMAIL') or input('Email: ')
password = os.getenv('TEST_PASSWORD') or getpass.getpass('Password: ')

token_url = f'{SUPABASE_URL}/auth/v1/token'
payload = { 'grant_type': 'password', 'email': email, 'password': password }
headers = { 'apikey': ANON_KEY, 'Content-Type': 'application/json' }

r = requests.post(token_url, json=payload, headers=headers)
print('status', r.status_code)
data = r.json() if r.content else {}
print('response keys:', list(data.keys()))
access_token = data.get('access_token') or data.get('access_token')
if not access_token:
    print('Falha ao obter access_token:', data)
else:
    print('Got access_token (first 40 chars):', access_token[:40])

In [ ]:
# 2) VALIDAR token via admin server (RECOMENDADO)
# Este endpoint usa a service key no servidor; o notebook não precisa da service key.
if not access_token:
    raise RuntimeError('Nenhum access_token disponível. Rode a célula de sign-in primeiro.')

verify_url = f'{ADMIN_SERVER}/admin/verify-session'
resp = requests.post(verify_url, json={'access_token': access_token})
print('verify-session status:', resp.status_code)
try:
    print('verify-session response:', resp.json())
except Exception:
    print('verify-session response text:', resp.text)

# Também é possível listar /admin/alunos (GET) se o usuário tiver permissão:
alunos_url = f'{ADMIN_SERVER}/admin/alunos'
resp2 = requests.get(alunos_url, headers={'Authorization': f'Bearer {access_token}'})
print('alunos status:', resp2.status_code)
try:
    print('alunos payload keys:', list(resp2.json().keys()))
    # mostrar 1 registro de exemplo (se existir)
    if resp2.ok:
        print(resp2.json().get('alunos', [])[:1])
except Exception:
    print('alunos response text:', resp2.text)

In [ ]:
# 3) VALIDAÇÃO DIRETA (apenas para DEV) — usa a service role key no notebook (não recomendado em produção)
if not SERVICE_KEY:
    print('SERVICE_KEY ausente; esta célula requer SUPABASE_SECRET_KEY no .env para testar diretamente.')
else:
    user_url = f'{SUPABASE_URL}/auth/v1/user'
    r = requests.get(user_url, headers={ 'Authorization': f'Bearer {access_token}', 'apikey': SERVICE_KEY })
    print('direct validation status:', r.status_code)
    try:
        print('user:', r.json())
    except Exception:
        print('response text:', r.text)

---
Próximos passos:
- Se a validação via `/admin/verify-session` funcionar, integre o mesmo fluxo ao seu frontend de produção.
- Se preferir, eu adiciono células adicionais que mostram como automatizar refresh tokens e recuperar `access_token` via OAuth.